In [1]:
import sqlite3

Eesmärk luua tabel, kus on iga verbi kohta (mis *vp_data3* andmebaasi mustrites esinevad) toodud ADVMOD esinemissagedus transaktsioonides koos verbiga ning verbi absoluutne esinemissagedus transaktsioonides, samuti relatiivne erinevus ja absoluutne erinevus verbi ja ADVMOD esinemissageduste vahel.

In [2]:
con = sqlite3.connect("C:/Users/liivas/Documents/Töö/verbirektisoonid/v32_data.db")
cur = con.cursor()

In [3]:
# mustrite andmebaasi lisamine
cur.execute('ATTACH DATABASE "C:/Users/liivas/Documents/Töö/verbirektisoonid/vp_data3.db" AS vp')

In [4]:
# uue andmebaasi lisamine ADVMOD esinemissageduste tabeli tarbeks
cur.execute('ATTACH DATABASE "advmod_count.db" AS adv')

Eesmärk luua tabel, kus on iga verbi kohta (mis mustritabelis esinevad) toodud ADVMOD esinemissagedus koos verbiga transaktsioonides ning verbi absoluutne esinemissagedus transatsioonides.

In [44]:
cur.execute("""
DROP TABLE IF EXISTS adv.advmod_count
""")

In [5]:
cur.execute("""
CREATE TABLE adv.advmod_count AS
SELECT
    tbl1.verb,
    tbl1.verb_compound,
    advmod_count,
    count(*) AS verb_count,
    (CAST(count(*) AS REAL) - CAST(advmod_count AS REAL)) / count(*) * 100 AS relative_diff,
    count(*) - advmod_count AS absolute_diff
FROM
(
    SELECT
        verb,
        verb_compound
    FROM
        vp.verb_matches as verbs
    INNER JOIN
        transaction_head as tr_head
    ON 
        verbs.head_id=tr_head.id
) as tbl1
INNER JOIN
(
    SELECT
        verb,
        verb_compound,
        count(*) AS advmod_count
    FROM
    (
        SELECT
            id,
            verb,
            verb_compound
        FROM
            vp.verb_matches as verbs2
        INNER JOIN
            transaction_head as tr_head2
        ON 
            verbs2.head_id=tr_head2.id
    ) AS tbl_verb
    INNER JOIN
    (
        SELECT DISTINCT
            head_id
        FROM
            transaction_row AS tr_row
        WHERE
            tr_row.deprel='advmod'
    ) AS tbl_advmod
    ON
        tbl_verb.id=tbl_advmod.head_id
    GROUP BY
        verb, verb_compound
) AS tbl2
ON
    tbl1.verb = tbl2.verb AND tbl1.verb_compound = tbl2.verb_compound
GROUP BY
    tbl1.verb, tbl1.verb_compound
ORDER BY
    relative_diff ASC
""")

In [7]:
con.close()